# 03 · Task3 pure-color validation

验证彩色/无色、红/绿和颜色调谐；Task3 不用于真假颜色比较。

In [ ]:
# [Setup]
from pathlib import Path
import sys, numpy as np, pandas as pd
ROOT=Path('/home/lirui/liulab_project/ieeg/Project_colorieeg_2026'); PIPE=ROOT/'color_cognition_pipeline'/'analyse_0720'
sys.path.insert(0,str(PIPE)); import config
from utils.epochs import load_epochs
from utils.decoding import make_time_windows, fixed_cv, decode_accuracy, decode_permutation

In [ ]:
# [Selection] Reuse the spatially defined candidates
cand=pd.read_csv(config.subject_result_dir('test001')/'localizer'/'candidate_electrodes_10mm.csv')
channel_col=next(c for c in cand.columns if c.lower() in ('channel','channelname','name'))
ep=load_epochs(config.INTERMEDIATE_ROOT/'test001'/'preprocessing'/'task3_erp.npz')
names=list(ep['channel_names']); picks=[names.index(c) for c in cand[channel_col].drop_duplicates() if c in names]
trig=np.char.replace(ep['triggers'].astype(str),'Trigger-In:',''); data=ep['data'][:,picks,:]

In [ ]:
# [Contrasts] Chromatic/achromatic response and red/green decoding
win=(ep['times_ms']>=80)&(ep['times_ms']<=300)
rows=[]
for code in ('51','52','53','54','55','56'):
    values=data[trig==code][:,:,win].mean((1,2)); rows.append({'trigger':code,'mean':values.mean(),'sem':values.std(ddof=1)/np.sqrt(len(values)),'n':len(values)})
summary=pd.DataFrame(rows); out=config.subject_result_dir('test001')/'task3'; summary.to_csv(out/'condition_response_80_300ms.csv',index=False)
mask=np.isin(trig,['51','54']); y=(trig[mask]=='54').astype(int); windows,centers=make_time_windows(ep['times_ms'],config.WINDOW_MS,config.STEP_MS)
cv=fixed_cv(y,config.N_SPLITS,config.RANDOM_SEED); score=decode_accuracy(data[mask],y,windows,cv)
pd.DataFrame({'time_ms':centers,'accuracy':score}).to_csv(out/'red_green_decoding.csv',index=False); display(summary)

In [ ]:
# [Plot] Display inline and save reusable figures
from utils.plotting import plot_condition_summary, plot_decoding
plot_condition_summary(summary,'Task3 candidate-electrode responses',out/'condition_response_80_300ms.png');
plot_decoding(centers,score,title='Task3 red vs green decoding',output=out/'red_green_decoding.png');